# RME v5 — data refinement, controlled adaptation, model selection
Scope: Indonesian medical dictation as a clinician-reviewed draft. Complete self-contained Colab notebook.
Research synthesis: `DEEP_RESEARCH_V5.md`; operating instructions: `README.md`.
Training GPU has not been run by the author. Small-model CPU integration and pure logic tests are provided.
Default runs TWO training profiles plus a 256-clip teacher audit; A100 recommended. This costs more time than v4.
No ZIP or separate Python upload needed. Existing v3/v4 split stays locked. No promise of an automatic accuracy improvement.

In [ ]:
import subprocess, sys
PACKAGES = ['transformers==4.57.3','peft==0.18.0','accelerate==1.12.0',
            'numpy==2.2.6','scipy==1.15.3','soundfile==0.13.1','librosa==0.11.0',
            'gdown==5.2.0','pytest==8.4.2','rapidfuzz==3.14.3','datasketch==1.6.5','silero-vad==6.2.0','onnxruntime==1.23.2']
subprocess.run([sys.executable,'-m','pip','install','-q',*PACKAGES],check=True)
# Jika Colab meminta restart, restart runtime lalu Run all. Torch/CUDA bawaan tidak diganti.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, pathlib, torch
from pathlib import Path
os.chdir('/content')
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab dahulu')
ROOT = Path('/content/gold_rme_v3')
WORK = Path('/content/drive/MyDrive/rme_asr_v3')
WORK.mkdir(parents=True,exist_ok=True)
ARCHIVE = Path('/content/gold_dataset.tar.gz')
FILE_ID = '1B551Fgb98EaK0Bx-g4BaCogzB3QjOjfR'
EXPECTED_ARCHIVE_SHA256 = '1213c8df5ca041dbf7dcfb119cd2d9112d970ad0849cff2193f02d1dd65c68d6'
SEED = 13
print(torch.cuda.get_device_name(0), 'torch',torch.__version__,'CUDA',torch.version.cuda)
V5=WORK/'v5';V5.mkdir(exist_ok=True)
# Two predeclared ablations by default; A100 is recommended. One-profile mode is cheaper.
PROFILE_NAMES=['decoder_qv','top8_qv']
BASE_MODEL_ID='openai/whisper-large-v3-turbo'  # optional 'openai/whisper-large-v3' requires a separate experiment
TEACHER_AUDIT_N=256  # training-only review prioritization; 0 disables, does not pretend fully purified data
MAX_RECORDING_CLIPS=300  # None disables the cap for a controlled ablation
SYNTHETIC_LIMIT=.05
NEGATIVE_FRACTION=.02
SELECTION_N,CONFIRM_N=600,400

In [ ]:
%%writefile rme_core.py
"""RME ASR v3: shared data contracts and transparent metrics. No clinical auto-correction."""
import collections, hashlib, json, math, re, tarfile, unicodedata
from pathlib import Path
import numpy as np

SR = 16000
BASE = 'openai/whisper-large-v3-turbo'

def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def write_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix(p.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    tmp.replace(p)

def normalize(text):
    # Keep decimals, dose separators, percent and negation. No synonym/filler rewriting.
    t = unicodedata.normalize('NFKC', str(text)).lower()
    t = re.sub(r'(?<=\d),(?=\d)', '.', t)
    t = re.sub(r'(?<!\d)\.|\.(?!\d)', ' ', t)
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s./%+\-]', ' ', t)).strip()

def extract_safe(archive, dest):
    dest = Path(dest).resolve(); dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive) as t:
        for m in t:
            p = (dest/m.name).resolve()
            if not p.is_relative_to(dest) or not (m.isfile() or m.isdir()):
                raise ValueError('Unsafe archive member: ' + m.name)
            t.extract(m, dest, filter='data')

def read_records(path):
    p = Path(path)
    if p.suffix == '.jsonl': return [json.loads(x) for x in p.read_text().splitlines() if x.strip()]
    if p.suffix == '.csv':
        import csv
        return list(csv.DictReader(p.open()))
    obj = json.loads(p.read_text())
    if isinstance(obj, list): return obj
    for key in ('data', 'records', 'samples', 'rows'):
        if isinstance(obj.get(key), list): return obj[key]
    raise ValueError(f'Unsupported manifest structure: {p}')

def canonical_rows(manifests, root):
    root = Path(root).resolve()
    files = collections.defaultdict(list)
    for p in root.rglob('*'):
        if p.suffix.lower() in ('.wav','.mp3','.flac','.ogg'): files[p.name].append(p)
    rows = []; seen = set()
    for manifest in manifests:
        for x in read_records(manifest):
            name = x.get('audio_file') or x.get('audio_path') or x.get('path') or x.get('file')
            text = x.get('raw_text', x.get('text', x.get('transcript', x.get('ref'))))
            if isinstance(text,str): text=re.sub(r'</?[A-Za-z][^>]*>', '', text)  # remove corpus language tags only
            if not name or not isinstance(text,str): raise ValueError('Manifest needs audio_file and text')
            name = str(name); candidates = [root/name, Path(manifest).parent/name]
            found = [p.resolve() for p in candidates if p.is_file() and p.resolve().is_relative_to(root)]
            if not found:
                found = files.get(Path(name).name, [])
                if len(found) != 1: raise ValueError('Missing/ambiguous audio: ' + name)
            p = found[0]; rel = str(p.relative_to(root))
            if rel in seen: raise ValueError('Duplicate manifest row: ' + rel)
            seen.add(rel)
            stem = p.stem
            syn = stem.startswith('SYN_') or str(x.get('source','')).lower() in ('synthetic','tts')
            source = 'synthetic' if syn else 'medisco' if stem.startswith('medisco_') else 'video' if re.match(r'(lvl3|clean)_chk_',stem) else 'fleurs' if stem.startswith('fleurs_') else str(x.get('source','unknown'))
            video = str(x.get('video_id') or '')
            if not video:
                match = re.match(r'(?:lvl3|clean)_chk_\d+_(.{11})$', stem)
                if match: video = match[1]  # YouTube IDs may contain underscores.
            speaker = str(x.get('speaker_id') or '')
            if source == 'medisco' and not speaker:
                match = re.search(r'(?:female|male)(?:-\d+)?',stem)
                if match: speaker = 'medisco:' + match[0]
            # Synthetic template family is training only; never use it as a human speaker ID.
            group = str(x.get('session_id') or video or speaker or x.get('group_id') or '')
            if syn: group = 'synthetic'
            if source=='fleurs':group='fleurs_unknown_speakers'  # per-clip IDs do not establish speaker independence
            if not group: raise ValueError('Missing recording/speaker/session group: ' + rel)
            declared = next((v for v in Path(rel).parts if v in ('train','dev','test')), 'unknown')
            clean=x.get('clean_text',text)
            changes=critical_proxy(text,clean)
            review_flags=[f'clean_changed_{k}' for k,v in changes.items() if v]
            if normalize(text)!=normalize(clean):review_flags.append('clean_lexical_change')
            rows.append(dict(id=rel, audio_file=rel, ref=text.strip(), source=source, synthetic=syn,
                training_only=syn or source=='fleurs',source_detail=x.get('source','unknown'),
                clean_text=clean,label_review_flags=review_flags,reference_policy='source_raw_without_language_tags',
                video_id=video, speaker_id=speaker, session_id=str(x.get('session_id') or ''),
                patient_id=str(x.get('patient_id') or ''), group_id=group, declared_split=declared,
                category=x.get('clinical_category',x.get('topic','unknown')),
                declared_seconds=float(x.get('duration',x.get('duration_seconds',0)) or 0),
                reference_verified=x.get('reference_verified') is True))
    if not rows: raise ValueError('Empty dataset')
    return sorted(rows,key=lambda r:r['id'])

def inspect_audio(row, root):
    import soundfile as sf
    r = dict(row); issues = []
    if any(f in row.get('label_review_flags',[]) for f in ('clean_changed_number','clean_changed_unit','clean_changed_negation')):
        issues.append('label_critical_disagreement_requires_audio_review')
    try:
        path = Path(root)/r['audio_file']
        x, sr = sf.read(path, dtype='float32', always_2d=True)
        r.update(sr=sr, channels=x.shape[1], seconds=len(x)/sr,
                 audio_sha256=file_hash(path), pcm_sha256=hashlib.sha256(x.tobytes()).hexdigest())
        r['peak'] = float(np.max(np.abs(x))) if len(x) else 0.
        r['rms'] = float(np.sqrt(np.mean(x*x))) if len(x) else 0.
        r['clip_fraction'] = float(np.mean(np.abs(x)>=.999)) if len(x) else 0.
        if not np.isfinite(x).all(): issues.append('nonfinite')
        if not 0.2 <= r['seconds'] <= 30.: issues.append('duration_out_of_window')
        if r.get('declared_seconds',0)>0 and abs(r['seconds']-r['declared_seconds'])>max(.5,.1*r['declared_seconds']):
            issues.append('duration_metadata_mismatch_requires_review')
        if r['rms'] < 1e-5 and r['ref']: issues.append('silent_with_label')
        if not r['ref']: issues.append('empty_label_requires_nonspeech_verification')
        r['warnings'] = []
        wps = len(normalize(r['ref']).split())/max(r['seconds'],.001)
        if wps < .6 or wps > 4.5: r['warnings'].append('unusual_label_rate')
        if r['clip_fraction'] > .001: r['warnings'].append('clipping')
        words = normalize(r['ref']).split()
        if len(words)>10 and len(set(words))/len(words)<.25: r['warnings'].append('repetition')
    except Exception as e: issues.append('decode_error:' + type(e).__name__)
    r['reject_reasons'] = issues
    return r

def grouped_split(rows, seed=13):
    """Connected components prevent shared recordings/speakers/patients/audio in different splits."""
    parent = list(range(len(rows)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    seen = {}
    for i,r in enumerate(rows):
        for key in ('group_id','video_id','speaker_id','session_id','patient_id','pcm_sha256'):
            val=r.get(key)
            if not val: continue
            k=(key,val)
            if k in seen: parent[find(i)] = find(seen[k])
            seen[k]=i
    members=collections.defaultdict(list)
    for i,r in enumerate(rows): members[find(i)].append(r)
    out=[]
    for group in members.values():
        identities=sorted({(key,str(r[key])) for r in group for key in ('group_id','video_id','speaker_id','session_id','patient_id') if r.get(key)})
        gid=digest(identities)  # stable when clips within the same known source are added/removed
        u=int(digest([seed,gid])[:12],16)/16**12
        split='train' if any(r.get('training_only',r['synthetic']) for r in group) else 'test' if u<.1 else 'dev' if u<.2 else 'train'
        out.extend(dict(r,split=split,cluster_id=gid) for r in group)
    # Ensure each sufficiently large non-synthetic source has at least one group per split.
    # This is metadata-only and deterministic; never use model scores to choose a split.
    strata=collections.defaultdict(dict)
    for r in out:
        if not r.get('training_only',r['synthetic']):strata[r['source'] if 'source' in r else 'unknown'][r['cluster_id']]=r['split']
    for source,assign in strata.items():
        if len(assign)<3:continue
        for needed in ('train','dev','test'):
            counts=collections.Counter(assign.values())
            if counts[needed]:continue
            candidates=sorted(k for k,v in assign.items() if counts[v]>1)
            chosen=candidates[0];assign[chosen]=needed
        for r in out:
            if r['cluster_id'] in assign:r['split']=assign[r['cluster_id']]
    # Exact duplicates kept once per partition, train never learns held-out long references.
    refs={s:{normalize(r['ref']) for r in out if r['split']==s and len(normalize(r['ref']).split())>=8} for s in ('dev','test')}
    kept=[]; dropped=[]; pcm_seen=set()
    for r in sorted(out,key=lambda r:({'test':0,'dev':1,'train':2}[r['split']],r['id'])):
        ref=normalize(r['ref']); reason=None
        if r['split']=='train' and ref in refs['dev']|refs['test']: reason='heldout_exact_text'
        if r['split']=='dev' and ref in refs['test']: reason='test_exact_text'
        pcm=(r.get('sr'),r.get('pcm_sha256'))
        if pcm in pcm_seen: reason='duplicate_pcm'
        pcm_seen.add(pcm)
        (dropped if reason else kept).append(dict(r,drop_reason=reason) if reason else r)
    for s in ('train','dev','test'):
        if not any(r['split']==s for r in kept): raise ValueError('Empty split; insufficient independent groups')
    return kept,dropped

def edit_counts(a,b):
    # deterministic S/D/I counts, O(len(a)*len(b)) time and O(len(b)) memory
    prev=[(j,0,0,j) for j in range(len(b)+1)]
    for i,x in enumerate(a,1):
        cur=[(i,0,i,0)]
        for j,y in enumerate(b,1):
            c,s,d,k=prev[j-1]; sub=(c+(x!=y),s+(x!=y),d,k)
            c,s,d,k=prev[j]; delete=(c+1,s,d+1,k)
            c,s,d,k=cur[-1]; insert=(c+1,s,d,k+1)
            cur.append(min((sub,delete,insert), key=lambda z:z[0]))
        prev=cur
    _,s,d,i=prev[-1]; return int(s),int(d),int(i)

def score(ref,hyp):
    a,b=normalize(ref),normalize(hyp)
    s,d,i=edit_counts(a.split(),b.split())
    from rapidfuzz.distance import Levenshtein
    char_err=Levenshtein.distance(a,b)
    return dict(s=s,d=d,i=i,n=len(a.split()),err=s+d+i,chars=len(a),char_err=char_err)

def paired_cluster_ci(rows, a, b, B=2000, seed=13):
    if set(a)!=set(b) or set(a)!={r['id'] for r in rows}: raise ValueError('Prediction coverage mismatch')
    clusters=collections.defaultdict(lambda:np.zeros(3))
    for r in rows:
        sa,sb=score(r['ref'],a[r['id']]),score(r['ref'],b[r['id']])
        clusters[r['cluster_id']]+=np.array([sa['err'],sb['err'],sa['n']])
    v=np.array(list(clusters.values())); total=v.sum(0)
    if not total[2]: raise ValueError('No reference words')
    result={'delta_wer_pp':float(100*(total[0]-total[1])/total[2]),'clusters':len(v),'unit':'connected_source_group','ci95':None}
    if len(v)<2: return result
    rng=np.random.default_rng(seed); boot=[]
    for _ in range(B):
        t=v[rng.integers(len(v),size=len(v))].sum(0)
        if t[2]>0: boot.append(100*(t[0]-t[1])/t[2])
    result['ci95']=np.quantile(boot,[.025,.975]).tolist();return result

def corpus_metrics(rows, hyps):
    if set(hyps)!={r['id'] for r in rows} or any(not isinstance(h,str) for h in hyps.values()):
        raise ValueError('All planned clips must have predictions; failures cannot disappear')
    totals=collections.Counter()
    for r in rows: totals.update(score(r['ref'],hyps[r['id']]))
    return dict(totals,wer=100*totals['err']/totals['n'] if totals['n'] else None,
                cer=100*totals['char_err']/totals['chars'] if totals['chars'] else None,clips=len(rows))

def critical_proxy(ref,hyp):
    # Screening only: no semantic scope/association inference or correctness certification.
    patterns={'number':r'\b\d+(?:[.,]\d+)?(?:/\d+)?\b',
              'unit':r'\b(?:mg|ml|mcg|kg|mmhg|gram|miligram|mililiter|persen|unit)\b',
              'negation':r'\b(?:tidak|tanpa|bukan|disangkal|negatif|belum)\b'}
    return {k:collections.Counter(re.findall(p,normalize(ref))) != collections.Counter(re.findall(p,normalize(hyp))) for k,p in patterns.items()}

def release_status(evidence):
    required=('independent_real_clinical_test','human_gold_references','critical_error_review',
              'silent_noise_test','long_form_test','latency_sla','subgroup_review','data_rights_review')
    missing=[k for k in required if evidence.get(k) is not True]
    return {'status':'NOT_READY' if missing else 'CANDIDATE_FOR_CONTROLLED_PILOT', 'missing':missing,
            'note':'Not automatic enterprise certification; clinician signs each RME draft.'}

def term_metrics(rows,hyps,terms):
    """Exact lexical occurrence scoring, not entity linking or semantic correctness."""
    terms=sorted({normalize(t) for t in terms if normalize(t)})
    if not terms:return {'status':'NOT_MEASURED','reason':'No clinician-approved lexicon supplied'}
    tp=fp=fn=0
    for r in rows:
        a,b=normalize(r['ref']),normalize(hyps[r['id']])
        for t in terms:
            p=r'(?<!\w)'+re.escape(t)+r'(?!\w)'
            na,nb=len(re.findall(p,a)),len(re.findall(p,b))
            tp+=min(na,nb);fp+=max(0,nb-na);fn+=max(0,na-nb)
    return {'tp':tp,'fp':fp,'fn':fn,'precision':tp/(tp+fp) if tp+fp else None,
            'recall':tp/(tp+fn) if tp+fn else None,'lexicon_size':len(terms),
            'note':'Lexical matches only; overlapping aliases must be curated before use.'}


def quarantine_conflicts(rows):
    groups=collections.defaultdict(list)
    for r in rows:
        if r.get('pcm_sha256'):groups[(r.get('sr'),r['pcm_sha256'])].append(r)
    for group in groups.values():
        if len({normalize(r['ref']) for r in group})>1:
            for r in group:
                reason='identical_audio_conflicting_references'
                if reason not in r['reject_reasons']:r['reject_reasons'].append(reason)
    return rows

In [ ]:
%%writefile rme_data.py
"""Versioned data refinement; annotations never inferred from an LLM."""
import collections,csv,json,math
from pathlib import Path
from rme_core import digest,normalize,critical_proxy,file_hash
from rme_decode import loop_flags

def apply_reviews(rows,path,allowed_split='train'):
    if not Path(path).exists():return rows,{'applied':0,'sha256':None}
    reviews=list(csv.DictReader(Path(path).open()));lookup={r['id']:r for r in rows};seen=set();edits={}
    for review in reviews:
        k=review['id']
        if k in seen or k not in lookup:raise ValueError('Duplicate/unknown review id: '+k)
        seen.add(k);r=lookup[k]
        if r['split']!=allowed_split:raise ValueError('Review cannot cross locked split: '+k)
        if review['audio_sha256']!=r['audio_sha256']:raise ValueError('Reviewed audio has changed')
        if review['adjudicated'].lower()!='true' or not review['reviewer_1'] or not review['reviewer_2'] or review['reviewer_1']==review['reviewer_2']:
            raise ValueError('Two independent reviewers and adjudication required')
        text=review['verified_verbatim'].strip()
        if not text:raise ValueError('Use separately verified nonspeech manifest for empty labels')
        edits[k]=text
    return [dict(r,ref=edits[r['id']],reference_verified=True,reference_policy='two_reviewer_adjudication') if r['id'] in edits else dict(r) for r in rows], {'applied':len(edits),'sha256':file_hash(path)}

def cap_sources(rows,max_per_group=300,synthetic_fraction=.05,seed=13):
    groups=collections.defaultdict(list)
    for r in rows:groups[r['cluster_id']].append(r)
    kept=[];removed=[]
    for group in groups.values():
        ordered=sorted(group,key=lambda r:(not r.get('reference_verified',False),digest([seed,r['id']])))
        # Synthetic often shares a single group for leakage protection; cap separately below.
        cap=len(ordered) if max_per_group is None or all(r['synthetic'] for r in ordered) else max_per_group
        kept.extend(ordered[:cap]);removed.extend(dict(r,curation_reason='recording_cap') for r in ordered[cap:])
    real=[r for r in kept if not r['synthetic']];syn=sorted([r for r in kept if r['synthetic']],key=lambda r:digest([seed,r['id']]))
    budget=int(len(real)*synthetic_fraction/max(1e-9,1-synthetic_fraction))
    removed.extend(dict(r,curation_reason='synthetic_cap') for r in syn[budget:])
    return sorted(real+syn[:budget],key=lambda r:r['id']),removed

def independent_dev(rows,seed=13,selection_fraction=.7):
    groups=sorted({r['cluster_id'] for r in rows},key=lambda k:digest([seed,k]))
    if len(groups)<4:raise ValueError('Need at least four dev source groups')
    n=min(len(groups)-2,max(2,int(len(groups)*selection_fraction)))
    select=set(groups[:n]);return [r for r in rows if r['cluster_id'] in select],[r for r in rows if r['cluster_id'] not in select]

def balanced_sample(rows,n,seed=13):
    g=collections.defaultdict(list)
    for r in sorted(rows,key=lambda r:digest([seed,r['id']])):g[r['cluster_id']].append(r)
    out=[]
    while g and len(out)<n:
        for k in sorted(list(g)):
            out.append(g[k].pop())
            if not g[k]:del g[k]
            if len(out)>=n:break
    return out

def teacher_review(rows,predictions):
    """Disagreement prioritizes human listening. Teacher is neither oracle nor replacement label."""
    from rme_core import edit_counts
    result=[]
    for r in rows:
        record=predictions[r['id']];hyp=record['hyp'];rw=normalize(r['ref']).split();hw=normalize(hyp).split()
        wer=100*sum(edit_counts(rw,hw))/max(1,len(rw));critical=critical_proxy(r['ref'],hyp)
        priority=3 if record['decoding']['needs_review'] else 2 if any(critical.values()) or wer>50 else 1 if wer>20 else 0
        result.append({'id':r['id'],'audio_sha256':r['audio_sha256'],'source':r.get('source_detail',r['source']),
             'reference_dataset':r['ref'],'teacher_hyp':hyp,'disagreement_wer':wer,'priority':priority,**critical})
    return sorted(result,key=lambda r:(-r['priority'],-r['disagreement_wer'],r['id']))

def check_new_training(new_rows,locked_rows):
    """Additional corpus must already conform to audited schema; no implicit split creation."""
    reserved=[r for r in locked_rows if r['split']!='train'];keys=('group_id','video_id','speaker_id','session_id','patient_id','pcm_sha256')
    blocked={k:{r[k] for r in reserved if r.get(k)} for k in keys}
    seen={r['id'] for r in locked_rows};texts={normalize(r['ref']) for r in reserved if len(normalize(r['ref']).split())>=8}
    for r in new_rows:
        if r['id'] in seen:raise ValueError('Duplicate additional id')
        seen.add(r['id'])
        if r.get('split')!='train' or not r.get('reference_verified') or not r.get('speaker_id') or not r.get('session_id'):
            raise ValueError('New corpus requires train-only, verified reference, speaker/session IDs')
        if r.get('commercial_use_reviewed') is not True:raise ValueError('New corpus use rights must be reviewed')
        if r.get('ref') and (r.get('adjudicated') is not True or not r.get('reviewer_1') or not r.get('reviewer_2') or r['reviewer_1']==r['reviewer_2']):
            raise ValueError('Additional speech requires two-reviewer adjudication')
        rel=Path(r['audio_file'])
        if rel.is_absolute() or '..' in rel.parts:raise ValueError('Additional audio path must stay inside ROOT')
        for k in keys:
            if r.get(k) and r[k] in blocked[k]:raise ValueError('Additional train overlaps heldout '+k)
        if normalize(r['ref']) in texts:raise ValueError('Additional reference overlaps heldout')
    return new_rows

def verified_negatives(path,root,locked_rows,max_count):
    if not Path(path).exists():return []
    from rme_core import inspect_audio
    items=json.loads(Path(path).read_text());out=[]
    for x in items:
        if x.get('human_verified_nonspeech') is not True or x.get('commercial_use_reviewed') is not True or x.get('ref')!='':
            raise ValueError('Negative must be human-verified real nonspeech with empty reference')
        if not x.get('reviewer_1') or not x.get('reviewer_2') or x['reviewer_1']==x['reviewer_2']:raise ValueError('Nonspeech requires two reviewers')
        rel=Path(x['audio_file'])
        if rel.is_absolute() or not (Path(root)/rel).resolve().is_relative_to(Path(root).resolve()):raise ValueError('Unsafe negative path')
        r=inspect_audio(dict(x,synthetic=False,reference_verified=True,split='train'),root)
        r['reject_reasons']=[v for v in r['reject_reasons'] if v!='empty_label_requires_nonspeech_verification']
        if r['reject_reasons']:raise ValueError('Invalid negative audio')
        out.append(r)
    check_new_training(out,locked_rows)
    return sorted(out,key=lambda r:digest(r['id']))[:max_count]

def near_duplicate_candidates(rows,threshold=.85,num_perm=64,max_pairs=10000):
    from datasketch import MinHash,MinHashLSH
    lsh=MinHashLSH(threshold=threshold,num_perm=num_perm);byid={r['id']:r for r in rows};pairs=[]
    for r in sorted(rows,key=lambda r:r['id']):
        words=normalize(r['ref']).split()
        if len(words)<8:continue
        shingles={' '.join(words[i:i+4]) for i in range(len(words)-3)}
        h=MinHash(num_perm=num_perm,seed=13)
        for sh in sorted(shingles):h.update(sh.encode())
        for other in sorted(lsh.query(h)):
            if byid[other]['split']!=r['split'] and len(pairs)<max_pairs:
                pairs.append({'id_a':other,'id_b':r['id'],'split_a':byid[other]['split'],'split_b':r['split'],
                              'status':'candidate_text_overlap_requires_review'})
        lsh.insert(r['id'],h)
    return {'pairs':pairs,'limit_reached':len(pairs)>=max_pairs,'method':'approximate word4gram MinHash LSH; not proof of duplicate audio'}

In [ ]:
%%writefile rme_profiles.py
"""Predeclared ablations; actual parameter counts are recorded per run."""
import re
PROFILES={
 'decoder_qv':{'r':16,'lr':1e-5,'dropout':.05,'encoder_top':0,'projections':['q_proj','v_proj']},
 'top8_qv':{'r':16,'lr':5e-6,'dropout':.05,'encoder_top':8,'projections':['q_proj','v_proj']},
 'all_attention':{'r':16,'lr':1e-5,'dropout':.05,'encoder_top':None,'projections':['q_proj','k_proj','v_proj','out_proj']}}

def lora_targets(model,profile):
    p=PROFILES[profile];depth=model.config.encoder_layers
    top=depth if p['encoder_top'] is None else min(depth,p['encoder_top'])
    targets=[]
    for name,module in model.named_modules():
        if name.rsplit('.',1)[-1] not in p['projections']:continue
        if '.decoder.layers.' in name:targets.append(name);continue
        match=re.search(r'encoder.layers.(\d+)\.',name)
        if match and int(match[1])>=depth-top:targets.append(name)
    if not targets:raise ValueError('No matching adaptation modules')
    return targets

def parameter_report(model):
    total=sum(p.numel() for p in model.parameters());train=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if not total or not train:raise ValueError('No trainable parameters')
    return {'total_parameters_including_adapters':total,'trainable_parameters':train,
            'trainable_percent':100*train/total,'frozen_percent':100*(total-train)/total,
            'trainable_names':[n for n,p in model.named_parameters() if p.requires_grad]}

In [ ]:
%%writefile rme_decode.py
"""Reference-blind generation checks and conservative VAD. No transcript rewriting."""
import re,zlib
import numpy as np
from rme_core import SR,normalize

DECODE={'language':'indonesian','task':'transcribe','num_beams':3,'do_sample':False,
        'max_length':448,'use_cache':True,'no_repeat_ngram_size':6,'forced_eos_token_id':None}

def loop_flags(text):
    words=normalize(text).split();flags=[]
    for n in range(1,7):
        for i in range(max(0,len(words)-n*8+1)):
            block=words[i:i+n]
            if all(words[i+j*n:i+(j+1)*n]==block for j in range(1,8)):
                flags.append('repeated_phrase');break
        if flags:break
    b=text.encode()
    if len(words)>=20 and len(b)/max(1,len(zlib.compress(b)))>2.4:flags.append('high_compression')
    return flags

def cap_hit(tokens,eos,pad,limit=448):
    # Count through FIRST EOS; padding cannot disguise a naturally early ending.
    ids=[int(x) for x in tokens]
    first=next((i for i,x in enumerate(ids[1:],1) if x==eos),len(ids))
    return first>=limit-1 or (len(ids)>=limit and eos not in ids[1:])

def choose_retry(primary,retry,primary_flags,retry_flags):
    return retry if primary_flags and not retry_flags else primary

class SpeechAudit:
    def __init__(self):
        from silero_vad import load_silero_vad
        self.model=load_silero_vad(onnx=True)
    def __call__(self,wave):
        import torch
        from silero_vad import get_speech_timestamps
        spans=get_speech_timestamps(torch.from_numpy(np.asarray(wave,dtype=np.float32)),self.model,
            sampling_rate=SR,threshold=.35,min_speech_duration_ms=100,
            min_silence_duration_ms=500,speech_pad_ms=300)
        return {'speech_seconds':sum(x['end']-x['start'] for x in spans)/SR,
                'spans':spans,'vad_no_speech':not spans}

def guarded_generate(model,proc,batch,decode=None):
    import torch
    cfg=dict(DECODE if decode is None else decode)
    with torch.inference_mode():tokens=model.generate(**batch,**cfg)
    raw=proc.batch_decode(tokens,skip_special_tokens=True)
    texts=list(raw);meta=[]
    for i,(t,seq) in enumerate(zip(raw,tokens)):
        flags=loop_flags(t)
        capped=cap_hit(seq,proc.tokenizer.eos_token_id,proc.tokenizer.pad_token_id,cfg.get('max_length',448))
        if capped:flags.append('generation_limit')
        entry={'raw_primary':t,'primary_flags':flags,'retry_used':False,'token_cap_hit':capped}
        if flags:
            retry_cfg=dict(cfg,num_beams=5,no_repeat_ngram_size=3,repetition_penalty=1.05)
            one={k:v[i:i+1] for k,v in batch.items()}
            with torch.inference_mode():retry_tokens=model.generate(**one,**retry_cfg)
            retry=proc.batch_decode(retry_tokens,skip_special_tokens=True)[0]
            rf=loop_flags(retry)
            if cap_hit(retry_tokens[0],proc.tokenizer.eos_token_id,proc.tokenizer.pad_token_id,retry_cfg.get('max_length',448)):rf.append('generation_limit')
            texts[i]=choose_retry(t,retry,flags,rf)
            entry.update(retry_text=retry,retry_flags=rf,retry_used=not rf)
        entry['needs_review']=bool(loop_flags(texts[i])) or (bool(flags) and not entry['retry_used'])
        meta.append(entry)
    return texts,tokens,meta

In [ ]:
%%writefile rme_eval.py
"""Comparable ASR, clinical proxy and group uncertainty; no certification from proxy scores."""
import collections
from rme_core import corpus_metrics,critical_proxy,paired_cluster_ci,normalize,edit_counts
from rme_decode import loop_flags

def evaluate_run(rows,run):
    hyps={k:v['hyp'] for k,v in run['records'].items()}
    m=corpus_metrics(rows,hyps)
    m['critical_proxy']={k:sum(critical_proxy(r['ref'],hyps[r['id']])[k] for r in rows) for k in ('number','unit','negation')}
    m['unresolved_decoding']=sum(v['decoding']['needs_review'] for v in run['records'].values())
    m['length_cap_primary']=sum(v['token_cap_hit'] for v in run['records'].values())
    m['loop_flags_final']=sum(bool(loop_flags(h)) for h in hyps.values())
    m['retry_count']=sum(bool(v['decoding']['primary_flags']) for v in run['records'].values())
    m['empty_ref_false_activations']=sum(bool(normalize(hyps[r['id']])) for r in rows if not normalize(r['ref']))
    m['verified_reference_clips']=sum(bool(r.get('reference_verified')) for r in rows)
    m['timing']=run['timing'];return m

def compare(rows,candidate,control,B=2000):
    cm,bm=evaluate_run(rows,candidate),evaluate_run(rows,control)
    ch={k:v['hyp'] for k,v in candidate['records'].items()};bh={k:v['hyp'] for k,v in control['records'].items()}
    paired=paired_cluster_ci(rows,ch,bh,B=B)
    passed=cm['wer']<bm['wer'] and cm['cer']<=bm['cer'] and cm['unresolved_decoding']==0 and all(cm['critical_proxy'][k]<=bm['critical_proxy'][k] for k in cm['critical_proxy'])
    return {'passed_internal_screen':passed,'candidate':cm,'control':bm,'paired':paired,
            'independent_clinical_evidence':False,'reference_warning':'Proxies against unverified labels do not establish clinical correctness'}

def paired_edit_analysis(ref,hyp):
    s,d,i=edit_counts(normalize(ref).split(),normalize(hyp).split())
    return {'substitution':s,'deletion':d,'insertion':i}

def wilson_upper(k,n,z=1.96):
    if not n:return None
    from math import sqrt
    p=k/n;return (p+z*z/(2*n)+z*sqrt(p*(1-p)/n+z*z/(4*n*n)))/(1+z*z/n)

def clinician_review(path,planned_ids):
    import csv
    from pathlib import Path
    if not Path(path).exists():return {'status':'NOT_MEASURED'}
    rows=list(csv.DictReader(Path(path).open()))
    if len(rows)!=len(planned_ids) or {r['id'] for r in rows}!=set(planned_ids):raise ValueError('Clinical review needs exact unique coverage')
    errors=opportunities=0
    for r in rows:
        if r['adjudicated'].lower()!='true' or not r['reviewer_1'] or not r['reviewer_2'] or r['reviewer_1']==r['reviewer_2']:raise ValueError('Incomplete clinician adjudication')
        e,n=int(r['critical_errors']),int(r['critical_opportunities'])
        if not 0<=e<=n:raise ValueError('Invalid critical error counts')
        errors+=e;opportunities+=n
    return {'status':'HUMAN_REVIEWED','critical_errors':errors,'critical_opportunities':opportunities,
            'rate':errors/opportunities if opportunities else None,
            'note':'Opportunity definition/severity must be preregistered. No independence assumption or certification inferred.'}

In [ ]:
%%writefile rme_train.py
"""GPU training helpers; imported only after Colab dependencies are installed."""
import math, random
from dataclasses import dataclass
from pathlib import Path
import numpy as np
from rme_core import SR, BASE, corpus_metrics

def load_wave(path):
    import soundfile as sf
    from scipy.signal import resample_poly
    x,sr=sf.read(path,dtype='float32',always_2d=True)
    if not np.isfinite(x).all() or not 0 < len(x)/sr <= 30.: raise ValueError('Invalid audio: '+str(path))
    x=x.mean(axis=1)
    if sr!=SR:
        g=math.gcd(sr,SR);x=resample_poly(x,SR//g,sr//g)
    return np.asarray(x,dtype=np.float32)

class Clips:
    def __init__(self,rows,root,augment=False): self.rows,self.root,self.augment=rows,Path(root),augment
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        return {'path':str(self.root/r['audio_file']),'text':r['ref'],'augment':self.augment}

@dataclass
class Collator:
    processor: object
    rng: object = None
    _worker_seed: object = None
    def __call__(self,items):
        import torch
        worker_seed=torch.initial_seed() % 2**32
        if self.rng is None or self._worker_seed!=worker_seed:
            self.rng=np.random.default_rng(worker_seed);self._worker_seed=worker_seed
        waves=[]; ids=[]
        for it in items:
            x=load_wave(it['path'])
            if it['augment']:
                if self.rng.random()<.3:
                    rms=np.sqrt(np.mean(x*x))
                    x=x+self.rng.normal(0,rms*10**(-self.rng.uniform(20,35)/20),len(x)).astype(np.float32)
                if self.rng.random()<.3: x=x*10**(self.rng.uniform(-3,3)/20)
                peak=np.max(np.abs(x))
                if peak>.99:x=x*(.99/peak)
            waves.append(x)
            seq=self.processor.tokenizer(it['text']).input_ids
            if seq[0]==self.processor.tokenizer.convert_tokens_to_ids('<|startoftranscript|>'): seq=seq[1:]
            if len(seq)>448: raise ValueError('Label >448 tokens: segment and realign; never truncate')
            ids.append({'input_ids':seq})
        batch=self.processor.feature_extractor(waves,sampling_rate=SR,return_tensors='pt',return_attention_mask=True)
        labels=self.processor.tokenizer.pad(ids,return_tensors='pt')
        batch['labels']=labels['input_ids'].masked_fill(labels.attention_mask.ne(1),-100)
        return dict(batch)

def build_model(revision,r=16,profile="decoder_qv",model_id=BASE):
    import torch
    from transformers import WhisperForConditionalGeneration
    from peft import LoraConfig,get_peft_model
    model=WhisperForConditionalGeneration.from_pretrained(model_id,revision=revision,torch_dtype=torch.float32,attn_implementation='sdpa')
    model.config.use_cache=False
    model.config.apply_spec_augment=False  # mild waveform augmentation only; ablate on dev
    model.generation_config.language='indonesian';model.generation_config.task='transcribe'
    model.generation_config.forced_decoder_ids=None
    model.generation_config.max_length=448
    model.generation_config.use_cache=True
    from rme_decode import DECODE
    for key,value in DECODE.items():setattr(model.generation_config,key,value)
    from rme_profiles import lora_targets,PROFILES
    return get_peft_model(model,LoraConfig(r=r,lora_alpha=2*r,lora_dropout=PROFILES[profile]['dropout'],
        target_modules=lora_targets(model,profile),bias='none'))

def make_trainer(model,proc,train,dev,root,out,config):
    import torch
    from transformers import Seq2SeqTrainer,Seq2SeqTrainingArguments,EarlyStoppingCallback
    def metrics(pred):
        p=pred.predictions[0] if isinstance(pred.predictions,tuple) else pred.predictions
        p=np.where(p<0,proc.tokenizer.pad_token_id,p)
        labels=np.where(pred.label_ids<0,proc.tokenizer.pad_token_id,pred.label_ids)
        refs=proc.batch_decode(labels,skip_special_tokens=True)
        hyps=proc.batch_decode(p,skip_special_tokens=True)
        m=corpus_metrics([{'id':str(i),'ref':t} for i,t in enumerate(refs)],{str(i):t for i,t in enumerate(hyps)})
        from rme_decode import loop_flags
        return {'wer':m['wer'],'cer':m['cer'],'loop_rate':sum(bool(loop_flags(h)) for h in hyps)/len(hyps)}
    steps=math.ceil(len(train)/(config['batch']*config['accum']))
    every=max(1,min(125,math.ceil(steps/4)))
    args=Seq2SeqTrainingArguments(output_dir=str(out),
        per_device_train_batch_size=config['batch'],per_device_eval_batch_size=max(1,config['batch']//2),
        gradient_accumulation_steps=config['accum'],num_train_epochs=config['epochs'],
        learning_rate=config['lr'],lr_scheduler_type='cosine',warmup_ratio=.05,weight_decay=.01,
        max_grad_norm=1.,bf16=config['bf16'],fp16=torch.cuda.is_available() and not config['bf16'],
        gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},
        eval_strategy='steps',save_strategy='steps',eval_steps=every,save_steps=every,
        save_total_limit=2,load_best_model_at_end=True,metric_for_best_model='wer',greater_is_better=False,
        predict_with_generate=True,generation_max_length=448,generation_num_beams=3,
        eval_accumulation_steps=8,logging_steps=25,report_to='none',
        dataloader_num_workers=2,dataloader_pin_memory=True,dataloader_persistent_workers=False,
        remove_unused_columns=False,label_names=['labels'],seed=config['seed'],data_seed=config['seed'],
        optim='adamw_torch',save_safetensors=True)
    return Seq2SeqTrainer(model=model,args=args,train_dataset=Clips(train,root,True),
        eval_dataset=Clips(dev,root,False),data_collator=Collator(proc),processing_class=proc,
        compute_metrics=metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=4)])

def smoke_gradients(trainer):
    import torch
    batch=trainer.data_collator([trainer.train_dataset[0]])
    batch={k:v.to(trainer.args.device) for k,v in batch.items()}
    trainer.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    trainer.model.train();trainer.model.zero_grad(set_to_none=True)
    with trainer.compute_loss_context_manager():loss=trainer.model(**batch).loss
    if not torch.isfinite(loss): raise RuntimeError('Nonfinite smoke loss')
    loss.backward()
    grads=[p.grad for n,p in trainer.model.named_parameters() if p.requires_grad and 'lora_' in n]
    if not grads or any(g is None or not torch.isfinite(g).all() for g in grads):
        raise RuntimeError('Missing/nonfinite LoRA gradient')
    if not any(torch.count_nonzero(g)>0 for g in grads): raise RuntimeError('All LoRA gradients zero')
    trainer.model.zero_grad(set_to_none=True)
    return float(loss.detach())

In [ ]:
%%writefile rme_bench.py
"""Reference-blind ASR inference, fingerprinted resume, explicit coverage and timing."""
import gc,time
from pathlib import Path
import numpy as np
from rme_core import BASE,SR,digest,file_hash,write_json
from rme_train import load_wave
from rme_decode import guarded_generate,SpeechAudit,DECODE,loop_flags

def model_fingerprint(path):
    p=Path(path)
    files=sorted(f for f in p.rglob('*') if f.is_file() and f.suffix in ('.json','.safetensors','.bin'))
    if not any(f.suffix in ('.safetensors','.bin') for f in files): raise ValueError('No model weights: '+str(p))
    return digest([(str(f.relative_to(p)),file_hash(f)) for f in files])

def infer(rows,root,target,revision,out,decode,batch_size=2,base_id=BASE):
    import torch
    from transformers import WhisperProcessor,WhisperForConditionalGeneration
    from peft import PeftModel
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    is_base=target==base_id
    adapter=not is_base and (Path(target)/'adapter_config.json').exists()
    fingerprint=revision if is_base else model_fingerprint(target)
    run_id=digest({'model':fingerprint,'revision':revision,'base_id':base_id,'decode':decode,
                   'data':[(r['id'],r['audio_sha256']) for r in rows],'batch_size':batch_size,
                   'engine':'transformers-4.57.3-fp16-sdpa','torch':torch.__version__,
                   'code':[file_hash(Path(__file__).parent/x) for x in ('rme_core.py','rme_train.py','rme_bench.py','rme_decode.py','rme_data.py','rme_profiles.py','rme_eval.py')],
                   'gpu':torch.cuda.get_device_name(0)})
    cache=out/(run_id+'.json')
    data=__import__('json').loads(cache.read_text()) if cache.exists() else {'run_id':run_id,'records':{},'complete':False}
    if data['complete']:
        if set(data['records'])!={r['id'] for r in rows}: raise ValueError('Corrupt cache coverage')
        return data
    proc=WhisperProcessor.from_pretrained(base_id if is_base or adapter else target,**({'revision':revision} if is_base or adapter else {}))
    model=WhisperForConditionalGeneration.from_pretrained(base_id if is_base or adapter else target,
        **({'revision':revision} if is_base or adapter else {}),torch_dtype=torch.float16,attn_implementation='sdpa')
    if adapter:model=PeftModel.from_pretrained(model,target).merge_and_unload()
    model=model.to('cuda').eval();model.config.use_cache=True;model.config.apply_spec_augment=False
    model.generation_config.forced_decoder_ids=None
    vad=SpeechAudit()
    def predict(waves):
        batch=proc.feature_extractor(waves,sampling_rate=SR,return_tensors='pt',return_attention_mask=True)
        batch={k:v.to('cuda',dtype=torch.float16 if k=='input_features' else v.dtype) for k,v in batch.items()}
        return guarded_generate(model,proc,batch,decode)
    pending=[r for r in rows if r['id'] not in data['records']]
    if pending:predict([load_wave(Path(root)/pending[0]['audio_file'])])
    torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
    for start in range(0,len(pending),batch_size):
        part=pending[start:start+batch_size]
        torch.cuda.synchronize();t0=time.perf_counter()
        waves=[load_wave(Path(root)/r['audio_file']) for r in part]
        vad_info=[vad(w) for w in waves]  # audit only; do not cut weak clinical words
        texts,tokens,diagnostics=predict(waves)
        torch.cuda.synchronize();elapsed=time.perf_counter()-t0
        for r,text,seq,diag,vi in zip(part,texts,tokens,diagnostics,vad_info):
            # Timing is batch wall time, never advertised as single-request latency.
            capped=diag['token_cap_hit']
            data['records'][r['id']]={'hyp':text,'seconds':r['seconds'],'batch_s':elapsed,
                'batch_n':len(part),'token_cap_hit':capped,'audio_sha256':r['audio_sha256'],'decoding':diag,'vad':vi}
        write_json(cache,data)
    # Batch=1 latency sample, deterministic and separate from throughput. Includes decode/preprocessing.
    lat=[];rtfs=[]
    sample=sorted(rows,key=lambda r:digest(r['id']))[:min(100,len(rows))]
    for r in sample:
        torch.cuda.synchronize();t0=time.perf_counter()
        wave=load_wave(Path(root)/r['audio_file']);vad(wave)
        predict([wave])
        torch.cuda.synchronize();dt=time.perf_counter()-t0
        lat.append(dt);rtfs.append(dt/r['seconds'])
    wall=sum(v['batch_s']/v['batch_n'] for v in data['records'].values())
    seconds=sum(r['seconds'] for r in rows)
    data.update(complete=True,hardware={'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'cuda':torch.version.cuda},timing={'warm_model_batch1_n':len(lat),'latency_p50_s':float(np.quantile(lat,.5)),
        'latency_p95_s':float(np.quantile(lat,.95)),'batch1_rtf_p95':float(np.quantile(rtfs,.95)),
        'batch_size':batch_size,'throughput_rtf':wall/seconds,'peak_allocated_gb':torch.cuda.max_memory_allocated()/2**30,
        'note':'Local audio decode + feature extraction + generation; includes VAD audit; excludes model load, queue, network, RME editing.'})
    write_json(cache,data)
    del model,proc;gc.collect();torch.cuda.empty_cache()
    return data

def challenge_infer(manifest, root, target, revision, out):
    """Optional independent real silence/noise/long-form cohort. HF overlap-chunked ASR, no raw truncation."""
    import json,math,soundfile as sf,torch
    from scipy.signal import resample_poly
    from transformers import WhisperProcessor,WhisperForConditionalGeneration,pipeline
    rows=json.loads(Path(manifest).read_text())
    if not rows or len({r['id'] for r in rows})!=len(rows):raise ValueError('Empty/duplicate challenge IDs')
    # Caller supplies independent approved cohort; this function cannot prove independence.
    proc=WhisperProcessor.from_pretrained(target)
    model=WhisperForConditionalGeneration.from_pretrained(target,torch_dtype=torch.float16,attn_implementation='sdpa')
    model.config.use_cache=True;model.config.apply_spec_augment=False;model.generation_config.forced_decoder_ids=None
    pipe=pipeline('automatic-speech-recognition',model=model,tokenizer=proc.tokenizer,
                  feature_extractor=proc.feature_extractor,device=0)
    results=[]
    for r in rows:
        if r['kind'] not in ('silence','noise','long_form','clinical'):raise ValueError('Unknown challenge kind')
        p=(Path(root)/r['audio_file']).resolve()
        if not p.is_relative_to(Path(root).resolve()):raise ValueError('Audio outside challenge root')
        x,sr=sf.read(p,dtype='float32',always_2d=True);x=x.mean(1)
        if not len(x) or not np.isfinite(x).all():raise ValueError('Invalid challenge audio')
        dur=len(x)/sr
        if r['kind'] in ('silence','noise') and r['ref'].strip():raise ValueError('Nonspeech reference must be empty')
        if r['kind']=='long_form' and dur<=30:raise ValueError('long_form must exceed 30 s')
        if sr!=SR:
            g=math.gcd(sr,SR);x=resample_poly(x,SR//g,sr//g)
        torch.cuda.synchronize();t=time.perf_counter()
        h=pipe({'raw':x,'sampling_rate':SR},chunk_length_s=30,stride_length_s=5,return_timestamps=True,
               generate_kwargs=DECODE)
        raw=h['text'];flags=loop_flags(raw)
        if flags:
            retry=pipe({'raw':x,'sampling_rate':SR},chunk_length_s=30,stride_length_s=5,return_timestamps=True,
                       generate_kwargs=dict(DECODE,num_beams=5,no_repeat_ngram_size=3,repetition_penalty=1.05))
            if not loop_flags(retry['text']):h=retry
        torch.cuda.synchronize()
        results.append(dict(r,hyp=h['text'],raw_primary=raw,needs_review=bool(loop_flags(h['text'])),seconds=dur,wall_s=time.perf_counter()-t,audio_sha256=file_hash(p)))
    write_json(out,{'records':results,'model':model_fingerprint(target),'base_revision':revision,
                    'protocol':'HF overlapping chunks 30s/5s; independent diagnostic; no streaming claim'})
    del pipe,model,proc;gc.collect();torch.cuda.empty_cache()
    return results

In [ ]:
%%writefile test_rme_core.py
import io,tarfile
import pytest
from rme_core import *

def test_numbers_and_negation_survive():
    assert normalize('Tidak, dosis 0,5 mg; TD 120/80.')=='tidak dosis 0.5 mg td 120/80'
    assert score('0,5 mg','5 mg')['err']==1
    assert score('tidak demam','demam')['d']==1

def test_empty_reference_insertions():
    m=corpus_metrics([{'id':'a','ref':''},{'id':'b','ref':'baik'}],{'a':'obat','b':'baik'})
    assert m['wer']==100 and m['i']==1

def test_no_silent_prediction_omission():
    with pytest.raises(ValueError): corpus_metrics([{'id':'a','ref':'obat'}],{})
    with pytest.raises(ValueError): corpus_metrics([{'id':'a','ref':'obat'}],{'a':None})

def test_cluster_pairing():
    rows=[{'id':str(i),'ref':'tidak demam','cluster_id':str(i//2)} for i in range(10)]
    a={r['id']:'tidak demam' for r in rows};b={r['id']:'demam' for r in rows}
    result=paired_cluster_ci(rows,a,b,B=100)
    assert result['clusters']==5 and result['ci95']==[-50.,-50.]
    with pytest.raises(ValueError): paired_cluster_ci(rows,a,{'x':'demam'})

def test_group_connected_components_and_synthetic_train_only():
    rows=[]
    for i in range(100):
        rows.append(dict(id=str(i),group_id='video'+str(i),speaker_id='speaker'+str(i//2),
             synthetic=False,pcm_sha256=str(i),sr=16000,ref='kata '+str(i)))
    rows.append(dict(id='syn',group_id='syn',speaker_id='',synthetic=True,pcm_sha256='syn',sr=16000,ref='sintetis'))
    out,dropped=grouped_split(rows)
    for key in ('group_id','speaker_id','pcm_sha256'):
        m={}
        for r in out:
            if r.get(key):
                assert m.setdefault(r[key],r['split'])==r['split']
    assert next(r for r in out if r['id']=='syn')['split']=='train'
    assert grouped_split(list(reversed(rows)))[0]==out

def test_dedup_heldout_text():
    rows=[dict(id=str(i),group_id=str(i),synthetic=False,pcm_sha256=str(i),sr=16000,
        ref='kalimat unik '+str(i)) for i in range(100)]
    # Duplicate speech joins same group; only one copy is retained.
    rows[1]['pcm_sha256']=rows[0]['pcm_sha256']
    out,drop=grouped_split(rows)
    assert len(out)==99 and drop[0]['drop_reason']=='duplicate_pcm'

def test_fail_closed_gate():
    assert release_status({})['status']=='NOT_READY'
    assert release_status({'human_gold_references':'True'})['status']=='NOT_READY'

def test_tar_traversal(tmp_path):
    p=tmp_path/'bad.tar'
    with tarfile.open(p,'w') as t:
        m=tarfile.TarInfo('../escape');m.size=1;t.addfile(m,io.BytesIO(b'x'))
    with pytest.raises(ValueError): extract_safe(p,tmp_path/'dest')

def test_youtube_underscore_group(tmp_path):
    audio=tmp_path/'train';audio.mkdir();(audio/'lvl3_chk_001_-qUTxy5a9_E.mp3').write_bytes(b'a')
    manifest=tmp_path/'manifest.json'
    write_json(manifest,[{'audio_file':'lvl3_chk_001_-qUTxy5a9_E.mp3','text':'nyeri'}])
    r=canonical_rows([manifest],tmp_path)[0]
    assert r['video_id']=='-qUTxy5a9_E'

def test_reject_ambiguous_basename(tmp_path):
    for s in ('train','test'):
        p=tmp_path/s;p.mkdir();(p/'same.wav').write_bytes(b'x')
    manifest=tmp_path/'manifest.json';write_json(manifest,[{'audio_file':'same.wav','text':'halo'}])
    with pytest.raises(ValueError):canonical_rows([manifest],tmp_path)

def test_numeric_proxy_not_semantic_certificate():
    assert critical_proxy('dosis 0,5 mg','dosis 5 mg')['number']
    assert critical_proxy('tidak demam','demam')['negation']
    assert not critical_proxy('tidak batuk, demam','batuk, tidak demam')['negation']

def test_term_precision_penalizes_invented_drug():
    m=term_metrics([{'id':'1','ref':'parasetamol'}],{'1':'parasetamol amoksisilin'},['parasetamol','amoksisilin'])
    assert m['recall']==1 and m['precision']==.5
    assert term_metrics([],{},[])['status']=='NOT_MEASURED'

def test_conflicting_same_audio_quarantines_both():
    rows=[{'sr':16000,'pcm_sha256':'same','ref':s,'reject_reasons':[]} for s in ('perut mulas','dada sesak')]
    assert all('identical_audio_conflicting_references' in r['reject_reasons'] for r in quarantine_conflicts(rows))

def test_manifest_uses_original_not_llm_rewrite(tmp_path):
    (tmp_path/'clean_chk_1_abcdefghijk.mp3').touch()
    p=tmp_path/'manifest.json'
    write_json(p,[{'audio_file':'clean_chk_1_abcdefghijk.mp3','raw_text':'tidak demam 0,5 mg',
        'clean_text':'demam 5 mg','group_id':'abcdefghijk','duration':3}])
    r=canonical_rows([p],tmp_path)[0]
    assert r['ref']=='tidak demam 0,5 mg' and r['declared_seconds']==3
    assert 'clean_changed_number' in r['label_review_flags']
    assert 'clean_changed_negation' in r['label_review_flags']

def test_audio_duration_mismatch_quarantined(tmp_path):
    import soundfile as sf
    sf.write(tmp_path/'x.wav',np.ones(16000,dtype=np.float32)*.1,16000)
    r=inspect_audio({'audio_file':'x.wav','ref':'nyeri dada','declared_seconds':2},tmp_path)
    assert 'duration_metadata_mismatch_requires_review' in r['reject_reasons']
    assert r['seconds']==1.

In [ ]:
%%writefile test_rme_decode.py
from rme_decode import loop_flags,cap_hit,choose_retry

def test_444_word_regression():
    assert 'repeated_phrase' in loop_flags('hai '*444)
    assert 'repeated_phrase' in loop_flags('selamat pagi '*80)

def test_valid_negation_not_removed():
    assert not loop_flags('tidak tidak dok, bukan lima, nol koma lima miligram')

def test_forced_eos_at_limit_is_cap():
    assert cap_hit([1]*447+[2],2,2,448)
    assert not cap_hit([1,3,2]+[2]*445,2,2,448)
    assert cap_hit([1]*448,2,2,448)

def test_retry_is_reference_blind_and_failed_text_preserved():
    assert choose_retry('hai '*444,'terima kasih',['loop'],[])=='terima kasih'
    assert choose_retry('raw','also bad',['loop'],['loop'])=='raw'
    assert choose_retry('good','different',[],[])=='good'

def test_stable_source_group_when_clip_added():
    from rme_core import grouped_split
    rows=[dict(id=str(i),group_id=str(i),synthetic=False,pcm_sha256=str(i),sr=16000,ref=str(i)) for i in range(100)]
    a,_=grouped_split(rows)
    more=dict(rows[1],id='extra',pcm_sha256='extra',ref='extra')
    b,_=grouped_split(rows+[more])
    assert {r['id']:r['split'] for r in a}=={r['id']:r['split'] for r in b if r['id']!='extra'}

def test_generate_retry_records_original(monkeypatch):
    import sys,types,contextlib,numpy as np
    monkeypatch.setitem(sys.modules,'torch',types.SimpleNamespace(inference_mode=contextlib.nullcontext))
    from rme_decode import guarded_generate,DECODE
    class Model:
        calls=0
        def generate(self,**kw):
            self.calls+=1
            return np.array([[1,5,2]]) if self.calls==1 else np.array([[1,6,2]])
    class Proc:
        tokenizer=types.SimpleNamespace(eos_token_id=2,pad_token_id=2)
        def batch_decode(self,seq,**kw):return ['hai '*444 if seq[0,1]==5 else 'terima kasih']
    m=Model();texts,tokens,meta=guarded_generate(m,Proc(),{'input_features':np.zeros((1,2))},DECODE)
    assert texts==['terima kasih'] and m.calls==2
    assert meta[0]['raw_primary']=='hai '*444 and meta[0]['retry_used'] and not meta[0]['needs_review']

def test_failed_retry_remains_visible(monkeypatch):
    import sys,types,contextlib,numpy as np
    monkeypatch.setitem(sys.modules,'torch',types.SimpleNamespace(inference_mode=contextlib.nullcontext))
    from rme_decode import guarded_generate
    model=types.SimpleNamespace(generate=lambda **kw:np.array([[1,5,2]]))
    proc=types.SimpleNamespace(tokenizer=types.SimpleNamespace(eos_token_id=2,pad_token_id=2),
        batch_decode=lambda seq,**kw:['hai '*444])
    text,_,meta=guarded_generate(model,proc,{'input_features':np.zeros((1,2))})
    assert text==['hai '*444] and meta[0]['needs_review'] and not meta[0]['retry_used']

In [ ]:
%%writefile test_rme_v5.py
import csv,json,types
import pytest
from rme_data import *
from rme_profiles import lora_targets,parameter_report
from rme_eval import clinician_review,wilson_upper

def row(i,**kw):
    r=dict(id=str(i),split='train',cluster_id=str(i//4),group_id=str(i//4),synthetic=False,ref='tidak demam',
           audio_sha256=str(i),pcm_sha256=str(i),source='video',reference_verified=False)
    r.update(kw);return r

def test_source_cap_and_synthetic_limit():
    data=[row(i) for i in range(100)]+[row(100+i,synthetic=True,cluster_id='syn') for i in range(100)]
    kept,removed=cap_sources(data,max_per_group=2,synthetic_fraction=.05)
    assert max(__import__('collections').Counter(r['cluster_id'] for r in kept if not r['synthetic']).values())<=2
    assert sum(r['synthetic'] for r in kept)/len(kept)<=.05
    assert len(kept)+len(removed)==len(data)

def test_dev_groups_disjoint():
    a,b=independent_dev([row(i,split='dev') for i in range(100)])
    assert not {r['cluster_id'] for r in a}&{r['cluster_id'] for r in b}
    assert len(a)+len(b)==100
    assert len(balanced_sample(a,11))==11

def test_two_reviewers_and_split_guard(tmp_path):
    p=tmp_path/'r.csv'
    fields=['id','audio_sha256','reviewer_1','reviewer_2','adjudicated','verified_verbatim']
    with p.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerow(dict(zip(fields,['1','1','a','b','true','tidak batuk'])))
    r,info=apply_reviews([row(1)],p)
    assert r[0]['ref']=='tidak batuk' and r[0]['reference_verified']
    with pytest.raises(ValueError):apply_reviews([row(1,split='test')],p)

def test_teacher_never_replaces_label():
    rr=[row(1)];before=rr[0]['ref']
    out=teacher_review(rr,{'1':{'hyp':'demam','decoding':{'needs_review':False}}})
    assert out[0]['priority']==2 and rr[0]['ref']==before

def test_additional_corpus_leak_is_rejected():
    held=row(1,split='test',speaker_id='doctorA')
    new=row(2,reference_verified=True,commercial_use_reviewed=True,speaker_id='doctorA',session_id='new')
    with pytest.raises(ValueError):check_new_training([new],[held])

def test_clinician_review_invalid_denominator(tmp_path):
    p=tmp_path/'c.csv'
    p.write_text('id,adjudicated,reviewer_1,reviewer_2,critical_errors,critical_opportunities\n1,true,a,b,2,1\n')
    with pytest.raises(ValueError):clinician_review(p,['1'])
    assert wilson_upper(0,0) is None and wilson_upper(0,100)>0

def test_lora_top_encoder_only_targets():
    names=[f'model.encoder.layers.{i}.self_attn.q_proj' for i in range(32)]+['model.decoder.layers.0.self_attn.q_proj','model.encoder.conv1']
    fake=types.SimpleNamespace(config=types.SimpleNamespace(encoder_layers=32),named_modules=lambda:[(k,None) for k in names])
    targets=lora_targets(fake,'top8_qv')
    assert len(targets)==9 and not any(f'layers.{i}.' in t for t in targets for i in range(1,24))
    assert lora_targets(fake,'decoder_qv')==['model.decoder.layers.0.self_attn.q_proj']

def test_rapidfuzz_character_count_preserves_existing_metric():
    from rme_core import score,edit_counts,normalize
    for a,b in [('tidak demam','demam'),('0,5 mg','5 mg'),('','obat'),('a b c','a c')]:
        assert score(a,b)['char_err']==sum(edit_counts(list(normalize(a)),list(normalize(b))))

def test_near_duplicate_audit_preserves_rows():
    data=[row(1,ref='pasien mengeluh nyeri dada sejak kemarin disertai batuk kering',split='train'),row(2,ref='pasien mengeluh nyeri dada sejak kemarin disertai batuk kering',split='test')]
    report=near_duplicate_candidates(data)
    assert len(report['pairs'])==1 and len(data)==2

def test_source_cap_can_be_disabled_for_ablation():
    data=[row(i,cluster_id='one') for i in range(500)]
    kept,removed=cap_sources(data,max_per_group=None)
    assert len(kept)==500 and not removed

In [ ]:
%%writefile test_tiny_training.py
"""Actual torch/PEFT/Whisper forward/backward test; no full model download or GPU."""
import contextlib,types
import numpy as np
import pytest
import torch
from transformers import WhisperConfig,WhisperForConditionalGeneration
from rme_profiles import parameter_report
from rme_train import build_model

@pytest.mark.parametrize('profile',['decoder_qv','top8_qv','all_attention'])
def test_real_tiny_whisper_lora_gradients(monkeypatch,profile):
    torch.set_num_threads(2);torch.manual_seed(13)
    cfg=WhisperConfig(vocab_size=64,num_mel_bins=8,d_model=16,encoder_layers=3,decoder_layers=2,
         encoder_attention_heads=2,decoder_attention_heads=2,encoder_ffn_dim=32,decoder_ffn_dim=32,
         max_source_positions=12,max_target_positions=20,pad_token_id=0,bos_token_id=1,eos_token_id=2,decoder_start_token_id=1,
         suppress_tokens=[],begin_suppress_tokens=[])
    monkeypatch.setattr(WhisperForConditionalGeneration,'from_pretrained',lambda *a,**kw:WhisperForConditionalGeneration(cfg))
    model=build_model('tiny',r=4,profile=profile)
    report=parameter_report(model)
    assert 0<report['trainable_percent']<100
    assert all('lora_' in n for n in report['trainable_names'])
    if profile=='decoder_qv':assert not any('.encoder.layers.' in n for n in report['trainable_names'])
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False});model.train()
    inputs=torch.randn(2,8,24);labels=torch.tensor([[3,4,2],[5,6,2]])
    loss=model(input_features=inputs,attention_mask=torch.ones(2,24,dtype=torch.long),labels=labels).loss
    assert torch.isfinite(loss)
    loss.backward();grads=[p.grad for p in model.parameters() if p.requires_grad]
    assert all(g is not None and torch.isfinite(g).all() for g in grads)
    assert any(torch.count_nonzero(g)>0 for g in grads)
    original={n:p.detach().clone() for n,p in model.named_parameters() if not p.requires_grad}
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-3);opt.step()
    for name,p in model.named_parameters():
        if name in original:assert torch.equal(p,original[name])
    model.eval();model.config.use_cache=True
    merged=model.merge_and_unload()
    assert not any('lora_' in n for n,p in merged.named_parameters())
    assert torch.isfinite(merged(input_features=inputs,labels=labels).loss)

def test_actual_trainer_step_and_generative_metrics(tmp_path,monkeypatch):
    import soundfile as sf
    from rme_train import make_trainer
    from transformers import EarlyStoppingCallback
    torch.set_num_threads(2)
    cfg=WhisperConfig(vocab_size=64,num_mel_bins=8,d_model=16,encoder_layers=2,decoder_layers=2,
         encoder_attention_heads=2,decoder_attention_heads=2,encoder_ffn_dim=32,decoder_ffn_dim=32,
         max_source_positions=12,max_target_positions=20,pad_token_id=0,bos_token_id=1,eos_token_id=2,
         decoder_start_token_id=1,suppress_tokens=[],begin_suppress_tokens=[])
    monkeypatch.setattr(WhisperForConditionalGeneration,'from_pretrained',lambda *a,**kw:WhisperForConditionalGeneration(cfg))
    model=build_model('tiny',r=4)
    model.generation_config.language=None;model.generation_config.task=None
    class Tokens:
        pad_token_id=0
        def __call__(self,text):return types.SimpleNamespace(input_ids=[1,3,2])
        def convert_tokens_to_ids(self,text):return 1
        def pad(self,ids,return_tensors):
            tensor=torch.tensor([x['input_ids'] for x in ids]);return types.SimpleNamespace(input_ids=tensor,attention_mask=torch.ones_like(tensor))
        def batch_decode(self,seq,**kw):return ['nyeri dada']*len(seq)
    class LabelBatch(dict):
        __getattr__=dict.__getitem__
    tok=Tokens();original_pad=tok.pad
    tok.pad=lambda *a,**kw:LabelBatch(vars(original_pad(*a,**kw)))
    class Features:
        def __call__(self,waves,**kw):return {'input_features':torch.zeros(len(waves),8,24),'attention_mask':torch.ones(len(waves),24,dtype=torch.long)}
    class Processor:
        tokenizer=tok;feature_extractor=Features();pad_token_id=0;eos_token_id=2;model_input_names=['input_features']
        def batch_decode(self,*a,**kw):return self.tokenizer.batch_decode(*a,**kw)
    sf.write(tmp_path/'a.wav',np.ones(8000,dtype=np.float32)*.1,16000)
    rows=[{'id':'a','ref':'nyeri dada','audio_file':'a.wav'}]
    trainer=make_trainer(model,Processor(),rows,rows,tmp_path,tmp_path/'out',
       {'batch':1,'accum':1,'epochs':1,'lr':1e-4,'bf16':False,'seed':13})
    trainer.args.max_steps=1;trainer.args.eval_strategy='no';trainer.args.save_strategy='no'
    trainer.args.load_best_model_at_end=False;trainer.args.dataloader_num_workers=0
    trainer.remove_callback(EarlyStoppingCallback)
    result=trainer.train();assert np.isfinite(result.training_loss)
    trainer.args.generation_max_length=10
    report=trainer.evaluate()
    assert report['eval_wer']==0. and report['eval_cer']==0.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','test_rme_core.py','test_rme_decode.py','test_rme_v5.py','test_tiny_training.py'],check=True)
from rme_core import *
from rme_data import *
from rme_eval import *
from rme_decode import SpeechAudit,loop_flags,DECODE
from rme_train import load_wave
from rme_bench import infer,model_fingerprint
from rme_profiles import PROFILES,parameter_report

## Data asli dan rencana split yang dibekukan

In [ ]:
import gdown
if not ARCHIVE.exists():
    result=gdown.download(id=FILE_ID,output=str(ARCHIVE),quiet=False,resume=True)
    if result is None: raise RuntimeError('Unduhan gagal; salin arsip ke ARCHIVE dari Drive')
archive_sha=file_hash(ARCHIVE)
if archive_sha!=EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError('Arsip berbeda/tidak lengkap. Periksa unduhan; audit ulang jika dataset sengaja berubah.')
marker=ROOT/'.archive_sha256'
if not marker.exists():
    extract_safe(ARCHIVE,ROOT)
    marker.write_text(archive_sha)
elif marker.read_text()!=archive_sha: raise RuntimeError('Gunakan ROOT baru untuk arsip berbeda')
# Manifest ditetapkan berdasarkan audit arsip, bukan rilis GitHub lama.
MANIFEST_NAMES = ['manifest_rme_train_gold.json', 'manifest_rme_dev_gold.json', 'manifest_rme_test_gold.json']
manifests=[]
for name in MANIFEST_NAMES:
    candidates=list(ROOT.rglob(name))
    if len(candidates)!=1:raise RuntimeError('Manifest tidak ditemukan/ambigu: '+name)
    manifests.extend(candidates)
rows=canonical_rows(manifests,ROOT)
print('Baris manifest:',len(rows),'| sumber:',dict(collections.Counter(r['source'] for r in rows)))

In [ ]:
from concurrent.futures import ThreadPoolExecutor
AUDIT_PATH=WORK/'audio_audit_v3.json'
audit_key=digest({'archive':archive_sha,'rows':rows,'schema':'v3.2'})
if AUDIT_PATH.exists():
    saved=json.loads(AUDIT_PATH.read_text())
    if saved['key']!=audit_key: raise RuntimeError('Audit cache berbeda; gunakan WORK baru')
    audited=saved['rows']
else:
    with ThreadPoolExecutor(max_workers=2) as ex:
        audited=list(ex.map(lambda r:inspect_audio(r,ROOT),rows))
    write_json(AUDIT_PATH,{'key':audit_key,'rows':audited})
audited=quarantine_conflicts(audited)
valid=[r for r in audited if not r['reject_reasons']]
rejected=[r for r in audited if r['reject_reasons']]
write_json(WORK/'rejected_audio.json',rejected)
print('Valid:',len(valid),'| rejected:',len(rejected))

In [ ]:
from huggingface_hub import HfApi
from transformers import WhisperProcessor,set_seed
set_seed(SEED)
REV_PATH=WORK/'base_revision.json'
if REV_PATH.exists():revision=json.loads(REV_PATH.read_text())['revision']
else:
    revision='41f01f3fe87f28c78e2fbf8b568835947dd65ed9'  # revision used in tokenizer audit
    write_json(REV_PATH,{'model':BASE,'revision':revision})
proc=WhisperProcessor.from_pretrained(BASE,revision=revision,language='indonesian',task='transcribe')
token_reject=[]; eligible=[]
for r in valid:
    ids=proc.tokenizer(r['ref']).input_ids
    if ids and ids[0]==proc.tokenizer.convert_tokens_to_ids('<|startoftranscript|>'):ids=ids[1:]
    (token_reject if len(ids)>448 else eligible).append(dict(r,label_tokens=len(ids)))
write_json(WORK/'rejected_tokens.json',token_reject)
planned,dedup=grouped_split(eligible,SEED)
write_json(WORK/'dedup_removed.json',dedup)
plan={'schema':'rme-v3.2','archive_sha256':archive_sha,'base_revision':revision,'seed':SEED,
      'rows':planned,'rejected_audio':len(rejected),'rejected_tokens':len(token_reject),'dedup_removed':len(dedup)}
plan['fingerprint']=digest(plan)
PLAN_PATH=WORK/'split_plan.json'
if PLAN_PATH.exists() and json.loads(PLAN_PATH.read_text())['fingerprint']!=plan['fingerprint']:
    raise RuntimeError('Split berbeda dari run sebelumnya; gunakan WORK baru')
write_json(PLAN_PATH,plan)
train_rows=[r for r in planned if r['split']=='train']
dev_all=[r for r in planned if r['split']=='dev']
# Round-robin group sampling so one long recording does not dominate checkpoint selection.
buckets=collections.defaultdict(list)
for r in sorted(dev_all,key=lambda r:digest(r['id'])):buckets[r['cluster_id']].append(r)
dev_rows=[]
while buckets and len(dev_rows)<600:
    for k in sorted(list(buckets)):
        dev_rows.append(buckets[k].pop())
        if not buckets[k]:del buckets[k]
        if len(dev_rows)==600:break
for s in ('train','dev','test'):
    rr=[r for r in planned if r['split']==s]
    print(s,len(rr),'clips',round(sum(r['seconds'] for r in rr)/3600,2),'hours',len({r['cluster_id'] for r in rr}),'groups')
print('Dev selection:',len(dev_rows),'| fingerprint:',plan['fingerprint'])

## Refine only train; separate selection and confirmation groups

In [ ]:
# Freeze plan semantics from v3/v4; change only training rows through documented refinement.
original_plan=plan
near_path=V5/'near_duplicate_candidates.json'
near_key=digest({'plan':plan['fingerprint'],'method':'word4gram-lsh-.85-64'})
if not near_path.exists():write_json(near_path,{'key':near_key,**near_duplicate_candidates(planned)})
elif json.loads(near_path.read_text())['key']!=near_key:raise RuntimeError('Near-dup audit mismatch')
# Conservative VAD audit on train only. Strong label/VAD disagreement is review quarantine.
vad=SpeechAudit();screen_path=V5/'training_screen.json'
screen_key=digest({'plan':plan['fingerprint'],'policy':'v5-vad-loop-1'})
if screen_path.exists():
    screen=json.loads(screen_path.read_text())
    if screen['key']!=screen_key:raise RuntimeError('Screen mismatch')
else:
    rejected=[]
    for r in train_rows:
        vv=vad(load_wave(ROOT/r['audio_file']))
        reasons=[]
        if vv['vad_no_speech']:reasons.append('no_speech_detected_requires_review')
        if loop_flags(r['ref']):reasons.append('repetitive_label_requires_review')
        if reasons:rejected.append(dict(id=r['id'],audio_sha256=r['audio_sha256'],reasons=reasons,vad=vv))
    screen={'key':screen_key,'rejected':rejected};write_json(screen_path,screen)
# Adjudication may fix labels on existing train IDs; it never changes test labels here.
reviewed,review_info=apply_reviews(planned,V5/'train_reviews.csv','train')
reviewed_ids={r['id'] for r in reviewed if r.get('reference_policy')=='two_reviewer_adjudication'}
excluded={r['id'] for r in screen['rejected']} - reviewed_ids
train_rows=[r for r in reviewed if r['split']=='train' and r['id'] not in excluded]
# Additional verified corpus is opt-in. See DATA_CONTRACT.md for audited schema and relative paths.
EXTRA=V5/'additional_train_audited.json'
if EXTRA.exists():
    extra=json.loads(EXTRA.read_text());check_new_training(extra,planned)
    for r in extra:
        aa=inspect_audio(r,ROOT)
        if aa['reject_reasons'] or aa['audio_sha256']!=r['audio_sha256'] or aa['pcm_sha256']!=r['pcm_sha256']:
            raise RuntimeError('Invalid additional audio: '+r['id'])
    train_rows+=extra
train_rows,cap_removed=cap_sources(train_rows,MAX_RECORDING_CLIPS,SYNTHETIC_LIMIT,SEED)
write_json(V5/'capped_training_rows.json',cap_removed)
negatives=verified_negatives(V5/'verified_nonspeech_train.json',ROOT,planned,max_count=int(len(train_rows)*NEGATIVE_FRACTION))
train_rows+=negatives
# Revalidate duplicate train audio/labels including added corpus. Never restore old test to train.
if len({r['id'] for r in train_rows})!=len(train_rows):raise RuntimeError('Duplicate training IDs')
seen_pcm={}
for r in train_rows:
    k=(r.get('sr'),r['pcm_sha256'])
    if k in seen_pcm:raise RuntimeError('Duplicate PCM in refined train; curate it first')
    seen_pcm[k]=r['ref']
ids={r['id'] for r in train_rows}
# All selected texts, including external corpus and human edits, need token preflight.
for r in train_rows:
    seq=proc.tokenizer(r['ref']).input_ids
    if seq and seq[0]==proc.tokenizer.convert_tokens_to_ids('<|startoftranscript|>'):seq=seq[1:]
    if len(seq)>448:raise RuntimeError('Refined label too long: '+r['id'])
# Separate dev source groups for model selection and confirmation.
selection_all,confirm_all=independent_dev(dev_all,SEED)
selection=balanced_sample(selection_all,SELECTION_N,SEED)
confirmation=balanced_sample(confirm_all,CONFIRM_N,SEED)
assert not {r['cluster_id'] for r in selection}&{r['cluster_id'] for r in confirmation}
# Optional adjudicated dev labels; neither selection nor confirmation is training data.
selection,sel_review=apply_reviews(selection,V5/'selection_dev_reviews.csv','dev')
confirmation,conf_review=apply_reviews(confirmation,V5/'confirmation_dev_reviews.csv','dev')
refinement={'original_plan':plan['fingerprint'],'training_rows':train_rows,'selection_ids':[r['id'] for r in selection],
 'confirmation_ids':[r['id'] for r in confirmation],'reviews':review_info,'selection_review':sel_review,'confirmation_review':conf_review,
 'schema':'rme-v5.1','synthetic_fraction_limit':SYNTHETIC_LIMIT,'max_recording_clips':MAX_RECORDING_CLIPS}
refinement['fingerprint']=digest(refinement)
write_json(V5/'refinement_plan.json',refinement)
print('Refined train:',len(train_rows),'selection:',len(selection),'confirmation:',len(confirmation))
print('Verified train labels:',sum(bool(r.get('reference_verified')) for r in train_rows),'| unverified labels remain provisional')

## Teacher audit for human review priority

In [ ]:
# Teacher observes TRAIN ONLY. No automatically substituted labels or disagreement-based test filtering.
from huggingface_hub import HfApi
if TEACHER_AUDIT_N>0:
    teacher_id='openai/whisper-large-v3'
    teacher_revision_path=V5/'teacher_revision.json'
    if not teacher_revision_path.exists():write_json(teacher_revision_path,{'model':teacher_id,'revision':HfApi().model_info(teacher_id).sha})
    teacher_rev=json.loads(teacher_revision_path.read_text())['revision']
    tt=balanced_sample([r for r in train_rows if r['ref']],TEACHER_AUDIT_N,SEED)
    tr=infer(tt,ROOT,teacher_id,teacher_rev,V5/'teacher_audit',DECODE,batch_size=1,base_id=teacher_id)
    priorities=teacher_review(tt,tr['records']);write_json(V5/'teacher_review_priority.json',priorities)
    import csv
    with (V5/'teacher_review_priority.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(priorities[0]));w.writeheader();w.writerows(priorities)
    with (V5/'train_review_template.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['id','audio_sha256','reviewer_1','reviewer_2','adjudicated','verified_verbatim'])
        for r in tt:w.writerow([r['id'],r['audio_sha256'],'','','',''])
    print('Teacher reviewed',len(tt),'train clips. Disagreement is review priority, not proof of wrong accent/reference.')

## Predeclared training profiles and real parameter counts

In [ ]:
from transformers import WhisperProcessor,set_seed,WhisperForConditionalGeneration
from transformers.trainer_utils import get_last_checkpoint
from rme_train import build_model,make_trainer,smoke_gradients
from peft import PeftModel
from huggingface_hub import HfApi
import importlib.metadata as im,gc,time
base_revision_path=V5/('revision_'+BASE_MODEL_ID.rsplit('/',1)[-1]+'.json')
if not base_revision_path.exists():
    rev=revision if BASE_MODEL_ID==BASE else HfApi().model_info(BASE_MODEL_ID).sha
    write_json(base_revision_path,{'model':BASE_MODEL_ID,'revision':rev})
base_rev=json.loads(base_revision_path.read_text())['revision']
proc=WhisperProcessor.from_pretrained(BASE_MODEL_ID,revision=base_rev,language='indonesian',task='transcribe')
versions={p:im.version(p) for p in ['torch','transformers','peft','accelerate','silero-vad','rapidfuzz','datasketch']}
mem=torch.cuda.get_device_properties(0).total_memory/2**30
# Full large-v3 has a larger decoder. Reduce microbatch; keep effective32.
batch=(8 if mem>=35 else 4 if mem>=22 else 2) if BASE_MODEL_ID==BASE else (2 if mem>=35 else 1)
base_selection=infer(selection,ROOT,BASE_MODEL_ID,base_rev,V5/'control_base_selection',DECODE,batch_size=1,base_id=BASE_MODEL_ID)
# Compare against actual v4 checkpoint as a stronger control when available.
V4_INFO=WORK/'latest_v4_candidate.json'
v4_info=json.loads(V4_INFO.read_text()) if V4_INFO.exists() else None
v4_selection=None
if v4_info:
    if v4_info['plan_fingerprint']!=plan['fingerprint']:raise RuntimeError('V4 split differs')
    if model_fingerprint(v4_info['model_dir'])!=v4_info['model_sha256']:raise RuntimeError('V4 weights changed')
    v4_selection=infer(selection,ROOT,v4_info['model_dir'],v4_info['revision'],V5/'control_v4_selection',DECODE,batch_size=1)
CANDIDATES=[]
for profile in PROFILE_NAMES:
    if profile not in PROFILES:raise RuntimeError('Unknown profile')
    recipe=PROFILES[profile]
    cfg=dict(recipe,profile=profile,batch=batch,accum=32//batch,epochs=3,seed=SEED,
        bf16=torch.cuda.get_device_capability(0)[0]>=8 and torch.cuda.is_bf16_supported())
    spec={'config':cfg,'refinement':refinement['fingerprint'],'base_id':BASE_MODEL_ID,'revision':base_rev,
          'versions':versions,'code':{p:file_hash(p) for p in ['rme_core.py','rme_train.py','rme_profiles.py','rme_data.py','rme_decode.py','rme_eval.py','rme_bench.py']}}
    run_id=digest(spec)[:16];RUN=V5/'runs'/run_id;RUN.mkdir(parents=True,exist_ok=True)
    write_json(RUN/'run_spec.json',spec)
    checkpoint=get_last_checkpoint(str(RUN))
    if checkpoint:
        for f in ['trainer_state.json','optimizer.pt','scheduler.pt','adapter_model.safetensors','rng_state.pth']:
            if not (Path(checkpoint)/f).exists():raise RuntimeError('Incomplete checkpoint: '+checkpoint)
    set_seed(SEED)
    model=build_model(base_rev,recipe['r'],profile,BASE_MODEL_ID)
    report=parameter_report(model);write_json(RUN/'parameter_report.json',report)
    print(profile,'trainable %:',report['trainable_percent'],'frozen %:',report['frozen_percent'])
    trainer=make_trainer(model,proc,train_rows,selection,ROOT,RUN,cfg)
    if not (RUN/'baseline_dev.json').exists():
        if checkpoint:raise RuntimeError('Baseline missing on resume')
        write_json(RUN/'baseline_dev.json',trainer.evaluate())
    print('Gradient smoke:',smoke_gradients(trainer));set_seed(SEED)
    started=time.perf_counter();result=trainer.train(resume_from_checkpoint=checkpoint)
    ADAPTER=RUN/'adapter';trainer.save_model(str(ADAPTER));proc.save_pretrained(ADAPTER)
    write_json(RUN/'training_result.json',{'metrics':result.metrics,'elapsed_this_session_s':time.perf_counter()-started,
       'best_checkpoint':trainer.state.best_model_checkpoint,'best_dev_wer':trainer.state.best_metric})
    trainer.model.to('cpu');del trainer,model;gc.collect();torch.cuda.empty_cache()
    base=WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_ID,revision=base_rev,torch_dtype=torch.float32)
    merged=PeftModel.from_pretrained(base,str(ADAPTER)).merge_and_unload()
    merged.config.use_cache=True;merged.config.apply_spec_augment=False
    for k,v in DECODE.items():setattr(merged.generation_config,k,v)
    MERGED=RUN/'merged';merged.to(torch.float16).save_pretrained(MERGED,safe_serialization=True);proc.save_pretrained(MERGED)
    del base,merged;gc.collect()
    pred=infer(selection,ROOT,str(MERGED),base_rev,RUN/'selection_eval',DECODE,batch_size=1,base_id=BASE_MODEL_ID)
    acceptance=compare(selection,pred,base_selection)
    if v4_selection:acceptance['versus_v4']=compare(selection,pred,v4_selection)
    passed=acceptance['passed_internal_screen'] and (acceptance.get('versus_v4',{}).get('passed_internal_screen',True))
    candidate={'run_id':run_id,'profile':profile,'model_dir':str(MERGED),'model_sha256':model_fingerprint(MERGED),
        'base_id':BASE_MODEL_ID,'revision':base_rev,'plan_fingerprint':plan['fingerprint'],
        'refinement_fingerprint':refinement['fingerprint'],'decode':DECODE,'parameter_report':report,
        'selection':acceptance,'selection_passed':passed,'enterprise_status':'NOT_READY'}
    write_json(RUN/'candidate.json',candidate);CANDIDATES.append(candidate)
write_json(V5/'candidate_comparison.json',CANDIDATES)

## Confirmation and candidate acceptance

In [ ]:
# Select ONCE on selection-dev, then check source-disjoint confirmation-dev.
# A failure does not trigger another candidate search on confirmation data.
passing=[c for c in CANDIDATES if c['selection_passed']]
chosen=min(passing or CANDIDATES,key=lambda c:c['selection']['candidate']['wer'])
write_json(V5/'selected_before_confirmation.json',chosen)
cc=infer(confirmation,ROOT,chosen['model_dir'],base_rev,V5/'confirmation_candidate',DECODE,batch_size=1,base_id=BASE_MODEL_ID)
bc=infer(confirmation,ROOT,BASE_MODEL_ID,base_rev,V5/'confirmation_base',DECODE,batch_size=1,base_id=BASE_MODEL_ID)
confirmation_result=compare(confirmation,cc,bc)
if v4_info:
    vc=infer(confirmation,ROOT,v4_info['model_dir'],v4_info['revision'],V5/'confirmation_v4',DECODE,batch_size=1)
    confirmation_result['versus_v4']=compare(confirmation,cc,vc)
accepted=chosen['selection_passed'] and confirmation_result['passed_internal_screen'] and confirmation_result.get('versus_v4',{}).get('passed_internal_screen',True)
chosen.update(confirmation=confirmation_result,accepted_internal=accepted,v4_control=v4_info,
    control_limitation='No v4 control available' if not v4_info else None)
write_json(V5/'latest_v5_candidate.json',chosen)
if accepted:write_json(V5/'latest_v5_accepted.json',chosen)
print('V5 INTERNAL:', 'PASS' if accepted else 'FAIL - keep previous model; candidate diagnostic only')
print('ENTERPRISE: NOT_READY; independent clinical validation required')